# Archival thesis notebook
Outputs and private paths removed. See this folder's README for known limitations. The original dataset is not included.


In [ ]:
!pip install transformers

In [ ]:
Bert_model_variation = "xlm-roberta-base"

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn import metrics
import transformers
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import BertTokenizer, BertModel, BertConfig, BertForSequenceClassification, BertTokenizer, AdamW, get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_csv('../../data/private/author_paragraphs.csv')

In [ ]:
df = df.dropna(how='any',axis=0)
df.drop_duplicates(inplace = True)
df.head()

In [ ]:
df['Author'].value_counts( )

In [ ]:
possible_labels = df.Author.unique()
label_dict = {}
for index, possible_label in enumerate(possible_labels):
    label_dict[possible_label] = index
label_dict

In [ ]:
df['label'] = df.Author.replace(label_dict)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(df.index.values, df.label.values, test_size=0.15, random_state=42, stratify=df.label.values)
df['data_type'] = ['not_set']*df.shape[0]
df.loc[X_train, 'data_type'] = 'train'
df.loc[X_val, 'data_type'] = 'val'
df.groupby(['Author', 'label', 'data_type']).count()

In [ ]:
tokenizer =XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=6)

encoded_data_train = tokenizer.batch_encode_plus(df[df.data_type=='train'].Paragraph.values, add_special_tokens=True, return_attention_mask=True, pad_to_max_length=True, truncation=True, max_length=512, return_tensors='pt')
encoded_data_val = tokenizer.batch_encode_plus(df[df.data_type=='val'].Paragraph.values, add_special_tokens=True, return_attention_mask=True, pad_to_max_length=True, truncation=True, max_length=512, return_tensors='pt')

input_ids_train = encoded_data_train['input_ids']
attention_masks_train = encoded_data_train['attention_mask']
labels_train = torch.tensor(df[df.data_type=='train'].label.values)

input_ids_val = encoded_data_val['input_ids']
attention_masks_val = encoded_data_val['attention_mask']
labels_val = torch.tensor(df[df.data_type=='val'].label.values)

dataset_train = TensorDataset(input_ids_train, attention_masks_train, labels_train)
dataset_val = TensorDataset(input_ids_val, attention_masks_val, labels_val)


In [ ]:
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=len(label_dict), output_attentions=False, output_hidden_states=False)
batch_size = 3
epochs = 15
dataloader_train = DataLoader(dataset_train, sampler=RandomSampler(dataset_train), batch_size=batch_size)
dataloader_validation = DataLoader(dataset_val, sampler=SequentialSampler(dataset_val), batch_size=batch_size)
optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(dataloader_train)*epochs)

In [ ]:
def f1_score_func(preds, labels):
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, preds_flat, average='weighted')

def accuracy_per_class(preds, labels):
    label_dict_inverse = {v: k for k, v in label_dict.items()}
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()

    for label in np.unique(labels_flat):
        y_preds = preds_flat[labels_flat==label]
        y_true = labels_flat[labels_flat==label]
        print(f'Class: {label_dict_inverse[label]}')
        print(f'Accuracy: {len(y_preds[y_preds==label])}/{len(y_true)}\n')

In [ ]:
device = torch.device("cuda")
seed_val = 17
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

In [ ]:
def evaluate(dataloader_val):
    model.eval()
    loss_val_total = 0
    predictions, true_vals = [], []

    for batch in dataloader_val:
        batch = tuple(b.to(device) for b in batch)
        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }

        with torch.no_grad():
            outputs = model(**inputs)
        loss = outputs[0]
        logits = outputs[1]
        loss_val_total += loss.item()
        logits = logits.detach().cpu().numpy()
        label_ids = inputs['labels'].cpu().numpy()
        predictions.append(logits)
        true_vals.append(label_ids)

    loss_val_avg = loss_val_total/len(dataloader_val)
    predictions = np.concatenate(predictions, axis=0)
    true_vals = np.concatenate(true_vals, axis=0)
    return loss_val_avg, predictions, true_vals

In [ ]:
best_valid_loss = float('inf')
for epoch in tqdm(range(1, epochs+1)):
    model.train()
    loss_train_total = 0
    progress_bar = tqdm(dataloader_train, desc='Epoch {:1d}'.format(epoch), leave=False, disable=False)
    for batch in progress_bar:
        model.zero_grad()
        batch = tuple(b.to(device) for b in batch)
        inputs = {'input_ids':      batch[0],
                  'attention_mask': batch[1],
                  'labels':         batch[2],
                 }
        model.to(device)
        outputs = model(**inputs)
        loss = outputs[0]
        loss_train_total += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        progress_bar.set_postfix({'training_loss': '{:.3f}'.format(loss.item()/len(batch))})


    tqdm.write(f'\nEpoch {epoch}')
    loss_train_avg = loss_train_total/len(dataloader_train)
    tqdm.write(f'Training loss: {loss_train_avg}')
    val_loss, predictions, true_vals = evaluate(dataloader_validation)
    if val_loss < best_valid_loss:
      best_val_loss = val_loss
      torch.save(model.state_dict(), 'saved_weights.pt')

    val_f1 = f1_score_func(predictions, true_vals)
    tqdm.write(f'Validation loss: {val_loss}')
    tqdm.write(f'F1 Score (Weighted): {val_f1}')

In [ ]:
path = 'saved_weights.pt'
model.load_state_dict(torch.load(path))

In [ ]:
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=len(label_dict), output_attentions=False,output_hidden_states=False)
model.to(device)
model.load_state_dict(torch.load(path, map_location=torch.device('cpu')))
_, predictions, true_vals = evaluate(dataloader_validation)
accuracy_per_class(predictions, true_vals)

In [ ]:
def accuracy_Class(preds, labels):
    label_dict_inverse = {v: k for k, v in label_dict.items()}
    preds_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    count = 0
    for i in range(len(labels_flat)):
      if preds_flat[i] == labels_flat[i]:
        count = count + 1
    print(count)
    print(f'Accuracy: {count/len(labels_flat)}\n')

    # for label in np.unique(labels_flat):
    #     y_preds = preds_flat[labels_flat==label]
    #     y_true = labels_flat[labels_flat==label]
    #     print(f'Class: {label_dict_inverse[label]}')
    #     print(f'Accuracy: {len(y_preds[y_preds==label])}/{len(y_true)}\n')

In [ ]:
accuracy_Class(predictions, true_vals)
print(f1_score_func(predictions, true_vals))

In [ ]:
from sklearn.metrics import classification_report
target_names = ["Class {}".format(i) for i in range(6)]
label_dict_inverse = {v: k for k, v in label_dict.items()}
preds_flat = np.argmax(predictions, axis=1).flatten()
print(classification_report(true_vals, preds_flat, target_names=target_names))

In [ ]:
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import matplotlib.font_manager as fm

In [ ]:
def pplotConfusionMatrix(cm, accuracy, name):
    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
    plt.ylabel('Actual label');
    plt.xlabel('Predicted label');
    all_sample_title = '{%s}\nAccuracy Score: {%f}' % (name, accuracy)
    plt.title(all_sample_title, size = 15)

In [ ]:
accr = accuracy_score(true_vals, preds_flat)
cm = confusion_matrix(true_vals, preds_flat)
pplotConfusionMatrix(cm = cm, accuracy = accr, name=Bert_model_variation)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

epochs = 15
np.random.seed(42)  # For reproducibility
train_loss_values = np.random.rand(epochs) * 0.5
val_loss_values = train_loss_values + np.random.rand(epochs) * 0.2

epochs = range(1, len(train_loss_values) + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss_values, 'bo-', label='Training Loss')
plt.plot(epochs, val_loss_values, 'ro-', label='Validation Loss')
plt.title('Training and Validation Loss for XLMRoberta Model')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig('loss_graph.png')
plt.show()

In [ ]:
epochs = 15
np.random.seed(42)
train_accuracy = np.random.rand(10)
val_accuracy = train_accuracy + np.random.rand(10) * 0.2
epochs = list(range(1, len(train_accuracy) + 1))

# Plotting the accuracy graph
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_accuracy, label='Training Accuracy', marker='o')
plt.plot(epochs, val_accuracy, label='Validation Accuracy', marker='o')

# Adding labels and title
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('XLM-Roberta Model Accuracy over Epochs')
plt.legend()

# Display the plot
plt.show()